<a href="https://colab.research.google.com/github/yasaswini1408/Prompt_Engineering/blob/main/Exp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-google-genai langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.9 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.1
  )

In [ ]:
llm_with_maps   = llm.bind_tools([{"google_maps": {}}])
llm_with_search = llm.bind_tools([{"google_search": {}}])

In [ ]:
maps_prompt = ChatPromptTemplate.from_template(
    "Locate the following venue and return its exact address, neighborhood, "
    "and operational hours and travel plan from vizag: {venue_query}"
    )
search_prompt = ChatPromptTemplate.from_template(
    "Search for recent local news, major public events, transit disruptions, "
    "or safety warnings happening today near: {venue_query}"
    )
aggregator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Elite Travel Concierge. Synthesize the Google Maps "
     "logistical data and the Google Search situational awareness "
     "data into a comprehensive evening briefing."),
      ("human", "Venue: {venue_query}\n\n[MAPS LOGISTICS]:\n{location_data}"
       "\n\n[SITUATIONAL NEWS & SAFETY]:\n{situational_data}") ]
                                          )

In [ ]:
maps_chain   = maps_prompt | llm_with_maps | StrOutputParser()
search_chain = search_prompt | llm_with_search | StrOutputParser()
parallel_grounding_engine = RunnableParallel(
    venue_query      = RunnablePassthrough(),
    location_data    = maps_chain,
    situational_data = search_chain
    )
travel_briefing_pipeline = (
    parallel_grounding_engine | aggregator_prompt | llm | StrOutputParser()
    )

In [ ]:
query = input("Enter the Topic: ")
print("--- Executing Parallel Grounding Pipeline (Maps + Search) ---\n")
briefing = travel_briefing_pipeline.invoke(query)
print(briefing)

Enter the Topic: Narendra Modi Cricket Stadium, Ahmedabad on May 31st, 2026
--- Executing Parallel Grounding Pipeline (Maps + Search) ---

## Evening Briefing: Narendra Modi Cricket Stadium, Ahmedabad - May 31st, 2026

**Good evening.** This briefing synthesizes logistical and situational data for your consideration regarding the **IPL 2026 Final** at the Narendra Modi Cricket Stadium in Ahmedabad on May 31st, 2026.

**Primary Event:** The main focus of the evening is the **IPL 2026 Final**, taking place at the Narendra Modi Cricket Stadium, located at Stadium Road, Motera, Sabarmati, Ahmedabad, Gujarat 380005. The stadium is highly rated (4.6 stars from 26,290 reviews) and offers wheelchair-accessible facilities. Paid parking is available on-site and on the street.

**Travel Considerations from Visakhapatnam (Vizag):**

*   **Air Travel:** This remains the most efficient option. Flights from Visakhapatnam Airport (VTZ) to Sardar Vallabhbhai Patel International Airport (AMD) in Ahmedab